# Associative Recall Inference on Pretrained Models

This notebook runs **inference only** on existing pretrained checkpoints using the sequence-length benchmark pipeline with `task_variant="associative_recall"`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.analysis import (
    compute_mean_rank_tables,
    nested_metric_table_to_long_df,
)
from pfns.experiments.model_benchmarks.evaluation import evaluate_models_over_seqlens
from pfns.experiments.model_benchmarks.model_registry import (
    get_autocast_models_from_registry,
    get_forward_models_from_registry,
)
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
from pfns.experiments.model_benchmarks.plotting import build_model_style_map, plot_curves_from_df


In [ ]:
EXPERIMENT = {
    "name": "ar_inference_pretrained",
    "num_classes": 10,
    "num_features": 10,
    "num_test_samples": 128,
    "num_repetitions": 500,
    "data_generation_seed": 42,
    "use_warmup_iters": False,
    "print_timing": False,
    "seqlen_list": [256, 512, 1024, 2048, 4096, 8192],
    "task_variant": "associative_recall",
    "task_kwargs": {},
}

LOCAL_AR_MODEL_REGISTRY = {
    "DeltaNet_Comb_ST": {
        "display_name": "DeltaNet_Comb_ST",
        "wandb_run_id": "associative_recall/runs/075sx98y",
        "eval_autocast_dtype": "bf16",
    },
    "Transformer_Comb_ST": {
        "display_name": "Transformer_ST",
        "wandb_run_id": "icl_arch/associative_recall/lkem4z49"
    },
    "GLA_Comb_ST": {
        "display_name": "GLA_Comb_ST",
        "wandb_run_id": "icl_arch/associative_recall/ki4uz4am",
    },
    "KDA_Comb_ST": {
        "display_name": "KDA_Comb_ST",
        "wandb_run_id": "icl_arch/associative_recall/rhggpvz6",
    },
    "Linear_Attentin_Comb_ST": {
        "display_name": "Linear_Attentin_Comb_ST",
        "wandb_run_id": "icl_arch/associative_recall/g1hqdmmi",
    },
    "Gated_DeltaNet_Comb_ST": {
        "display_name": "Gated_DeltaNet_Comb_ST",
        "wandb_run_id": "icl_arch/associative_recall/fxncltfq",
    },
    "DeltaNet_1k_steps_Comb_ST": {
        "display_name": "DeltaNet_1k_steps_Comb_ST",
        "wandb_run_id": "icl_arch/associative_recall/etor7x8b",
        "eval_autocast_dtype": "bf16",
    },
}

models_to_compare = {
    name: config.copy()
    for name, config in LOCAL_AR_MODEL_REGISTRY.items()
}

print(f"Selected {len(models_to_compare)} model(s):")
for name in models_to_compare:
    print(" -", name)

display(pd.DataFrame.from_dict(models_to_compare, orient="index"))


In [ ]:
device = str(get_default_device())
print("Using device:", device)

models, configs = load_models_for_benchmark(models_to_compare, device=device)
autocast_models = get_autocast_models_from_registry(models_to_compare, device=device)
forward_models = get_forward_models_from_registry(models_to_compare)

print("Loaded models:", list(models.keys()))
print("Autocast models:", list(autocast_models.keys()))
print("Forward-only models:", forward_models)

In [ ]:
results = evaluate_models_over_seqlens(
    models=models,
    configs=configs,
    seqlen_list=EXPERIMENT["seqlen_list"],
    num_features=EXPERIMENT["num_features"],
    num_classes=EXPERIMENT["num_classes"],
    number_of_test_samples=EXPERIMENT["num_test_samples"],
    number_of_repetitions=EXPERIMENT["num_repetitions"],
    use_warmup_iters=EXPERIMENT["use_warmup_iters"],
    print_timing=EXPERIMENT["print_timing"],
    autocast_models=autocast_models,
    forward_models=forward_models,
    device=device,
    data_generation_seed=EXPERIMENT["data_generation_seed"],
    task_variant=EXPERIMENT["task_variant"],
    task_kwargs=EXPERIMENT["task_kwargs"],
    progress_desc="AR inference",
)

results["metadata"]

In [ ]:
metric_df = nested_metric_table_to_long_df(results["metric_table"])
timing_df = nested_metric_table_to_long_df(results["timing_table"])
memory_df = nested_metric_table_to_long_df(results["memory_table"])

print("metric_df:", metric_df.shape)
print("timing_df:", timing_df.shape)
print("memory_df:", memory_df.shape)

display(metric_df.head())
display(timing_df.head())
display(memory_df.head())

In [ ]:
metric_summary = (
    metric_df.groupby(["model", "metric"], observed=True)["value"]
    .agg(mean="mean", std="std")
    .reset_index()
)
display(metric_summary.sort_values(["metric", "mean"], ascending=[True, False]))

rank_tables = compute_mean_rank_tables(metric_df)
display(rank_tables["overall_ranks"].sort_values(["metric", "rank"]))

In [ ]:
model_names = sorted(
    set(metric_df["model"].unique())
    | set(timing_df["model"].unique())
    | set(memory_df["model"].unique())
)
style_map = build_model_style_map(model_names)

PLOT_SPECS = {
    "metric": [
        ("acc", "Accuracy"),
        ("ce", "Cross Entropy Loss"),
        ("roc_auc", "ROC AUC"),
    ],
    "timing": [
        ("forward_time_ms", "Total Time (ms)"),
        ("fit_time_ms", "Fit Time (ms)"),
        ("predict_time_ms", "Predict Time (ms)"),
    ],
    "memory": [
        ("peak_allocated_mb", "Peak Allocated MB"),
        ("peak_reserved_mb", "Peak Reserved MB"),
        ("context_size_mb", "Context Size MB"),
    ],
}

PLOT_PANEL_CONFIGS = [
    ("metric", metric_df, {"log_x": True, "title_suffix": " vs Number of In-context Samples"}),
    ("timing", timing_df, {"show_std": True, "log_x": False, "title_suffix": " vs Number of In-context Samples"}),
    ("memory", memory_df, {"log_x": False, "title_suffix": " vs Number of In-context Samples"}),
]

def _available_specs(df, specs, *, panel_name):
    available = set(df["metric"].unique()) if not df.empty else set()
    selected = [(metric, label) for metric, label in specs if metric in available]
    missing = [metric for metric, _ in specs if metric not in available]
    if missing:
        print(f"Skipping missing {panel_name} metrics: {missing}")
    return selected

def plot_panel(df, specs, *, panel_name, value_col="value", show_std=False, log_x=False, invert_y=False, title_suffix=""):
    selected_specs = _available_specs(df, specs, panel_name=panel_name)
    if not selected_specs:
        print(f"No plottable metrics found for panel={panel_name!r}.")
        return None, None
    return plot_curves_from_df(
        df,
        specs=selected_specs,
        style_map=style_map,
        x_col="seqlen",
        value_col=value_col,
        x_label="Number of In-context Samples",
        title_suffix=title_suffix,
        show_std=show_std,
        log_x=log_x,
        invert_y=invert_y,
    )


In [ ]:
for panel_name, df, opts in PLOT_PANEL_CONFIGS:
    plot_panel(df, PLOT_SPECS[panel_name], panel_name=panel_name, **opts)
